#Task-1

In [1]:
df_fe['driver_to_rest_km'] = haversine_km(
    df_fe['Driver_Lat'], df_fe['Driver_Lon'],
    df_fe['Restaurant_Lat'], df_fe['Restaurant_Lon']
)

NameError: name 'haversine_km' is not defined

#Task-2


In [ ]:
df_fe['is_peak_hour'] = df_fe['order_hour'].isin([13, 14, 19, 20, 21]).astype(int)

# Task-3

In [2]:
for k in [10, 30, 50]:
    top_items = df_fe['Item_Name'].value_counts().head(k).index
    df_fe['Item_Name_reduced'] = np.where(df_fe['Item_Name'].isin(top_items), df_fe['Item_Name'], 'Other')

    X_temp = df_fe.drop(columns=drop_cols + [target_col])
    y_temp = df_fe[target_col]

    X_tr, X_te, y_tr, y_te = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp)

    cat_cols = X_tr.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = X_tr.select_dtypes(include=[np.number, "bool"]).columns.tolist()

    prep = ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ])

    model_temp = Pipeline(steps=[
        ("preprocess", prep),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample"))
    ])

    model_temp.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, model_temp.predict(X_te))
    print(f"top_k = {k} | Accuracy: {acc:.4f}")

    ohe = model_temp.named_steps["preprocess"].named_transformers_["cat"]
    cat_fn = ohe.get_feature_names_out(cat_cols)
    all_fn = np.concatenate([cat_fn, np.array(num_cols)])

    fi = pd.DataFrame({"feature": all_fn, "importance": model_temp.named_steps["rf"].feature_importances_})
    fi = fi.sort_values("importance", ascending=False)
    print(fi.head(5))
    print("-" * 40)

NameError: name 'df_fe' is not defined

#Task-4

In [ ]:
from sklearn.feature_selection import SelectFromModel

feature_selector = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))

model_fs = Pipeline(steps=[
    ("preprocess", preprocess),
    ("feature_selection", feature_selector),
    ("rf", rf)
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)
print("Accuracy with Feature Selection:", round(accuracy_score(y_test, y_pred_fs), 4))